# Multiclass product, channel, both, and calibration
Every model scope executes one CatBoost Optuna trial. A separate both-scope workflow enables one-vs-rest rolling calibration and saves raw/calibrated reports.

In [1]:
from pathlib import Path
import numpy as np
from fmlib.automl import MulticlassTask, MulticlassTaskConfig
ROOT=Path.cwd().parent.resolve(); data=ROOT/'data/multiclass'; cat=[f'cat_feature_{i}' for i in range(1,6)]; num=[f'num_feature_{i}' for i in range(1,6)]
rows=[]; expected_model_counts={'product':1,'channel':2,'both':3}
for scope in ('product','channel','both'):
 c=MulticlassTaskConfig(target_column='target', client_id_column='epk_id', group_column='group', treatment_column='treatment', inverse_treatment=True, report_month_column='report_month', environment={}, env_type='local',backend='boosting',engine='catboost',device='gpu',model_scope=scope,categorical_columns=cat,numerical_columns=num,hidden_state_columns=['seq_hidden_state'],hyperopt=True,n_trials=1,optimization_metric='roc_auc_ovr_macro',search_space={'iterations':[16],'depth':[2]},verbose=False,output_dir=ROOT/f'outputs/notebook_tests/multiclass/scopes/{scope}')
 t=MulticlassTask(c); training=t.train(data/'train',data/'valid'); assert len(training.best_params)==expected_model_counts[scope]; prediction=t.predict(data/'test'); evaluation=t.evaluate(data/'test',prediction); artifact=t.save(); rows.append((scope,len(training.best_params),evaluation.metrics['roc_auc_ovr_macro'],prediction.raw_scores is None,artifact.exists()))
calibrated_config=MulticlassTaskConfig(target_column='target', client_id_column='epk_id', group_column='group', treatment_column='treatment', inverse_treatment=True, report_month_column='report_month', environment={}, env_type='local',backend='boosting',engine='catboost',device='gpu',model_scope='both',categorical_columns=cat,numerical_columns=num,hidden_state_columns=['seq_hidden_state'],hyperopt=True,n_trials=1,optimization_metric='roc_auc_ovr_macro',search_space={'iterations':[16],'depth':[2]},verbose=False,calibration_windows={'2024-10-01':['2024-07-01','2024-08-01','2024-09-01'],'2024-11-01':['2024-08-01','2024-09-01'],'2024-12-01':['2024-09-01']},output_dir=ROOT/'outputs/notebook_tests/multiclass/scopes/both_calibrated')
calibrated=MulticlassTask(calibrated_config); training=calibrated.train(data/'train',data/'valid'); prediction=calibrated.predict(data/'test',calibration_paths=[data/'train',data/'valid']); evaluation=calibrated.evaluate(data/'test',prediction); artifact=calibrated.save(); score_cols=[f'score_{i}' for i in range(len(prediction.class_order))]; assert prediction.raw_scores is not None and np.allclose(prediction.scores.select(score_cols).sum_horizontal(),1.0); assert 'metrics_calibrated' in evaluation.excel_paths; rows.append(('both_calibrated',len(training.best_params),evaluation.calibrated_metrics['roc_auc_ovr_macro'],False,artifact.exists()))
rows


2026-08-17 14:41:03,167 INFO run_id=684e98a2d0994791958662c835f1900e Starting action=train task_config=MulticlassTaskConfig backend=boosting engine=catboost env_type=local


<workspace>\fmlib-main\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-08-17 14:41:05,605] A new study created in memory with name: no-name-ffcd178c-4b43-42a6-b106-588360a5c13e


[I 2026-08-17 14:41:08,771] Trial 0 finished with value: 0.729800552699325 and parameters: {'iterations': 16, 'depth': 2}. Best is trial 0 with value: 0.729800552699325.


2026-08-17 14:41:08,775 INFO run_id=684e98a2d0994791958662c835f1900e Finished action=train


2026-08-17 14:41:08,780 INFO run_id=e7809789463043408e7ef0831b74fb84 Starting action=predict task_config=MulticlassTaskConfig backend=boosting engine=catboost env_type=local


...

2026-08-17 14:41:40,452 INFO run_id=1d699505617f408f8ab18d14fe6fda41 Starting action=predict task_config=MulticlassTaskConfig backend=boosting engine=catboost env_type=local


2026-08-17 14:41:40,882 INFO run_id=1d699505617f408f8ab18d14fe6fda41 Finished action=predict


2026-08-17 14:41:40,890 INFO run_id=d1933824d5cc4dc08813e5e3f02007b2 Starting action=evaluate task_config=MulticlassTaskConfig backend=boosting engine=catboost env_type=local


2026-08-17 14:41:44,768 INFO run_id=d1933824d5cc4dc08813e5e3f02007b2 Finished action=evaluate


[('product', 1, 0.7354453945582575, True, True),
 ('channel', 2, 0.7354926595412854, True, True),
 ('both', 3, 0.7354926595412854, True, True),
 ('both_calibrated', 3, 0.7354094744579711, False, True)]